# Noisy-isomorphism criterion: k-flip vs independent pairs

For a source graph $A$, $A'$ is an **alignment-preserving** perturbation ($A' \in \mathcal N(A)$, *labelled*) if
$$\|A-A'\|_1 < \min_{P\notin\operatorname{Aut}(G)} \|PAP^T-A'\|_1 .$$
For unlabelled graphs, $G_{A'} \in \mathcal N(G_A)$ if **some** relabelling $PA'P^T \in \mathcal N(A)$.

Both conditions are checked exactly by brute force over all $n!$ permutations. Writing $O=\arg\min_S\|A-SA'S^T\|_1$ (with minimum value = GED):
* **unlabelled** holds $\iff |O| = |\operatorname{Aut}(G)|$ (optimal alignment unique up to automorphisms);
* **labelled** holds $\iff$ unlabelled holds **and** $\|A-A'\|_1 = \text{GED}$.

Groups: $(A',A)$ with $k\in\{1,2,3\}$ random edge flips, or $A'$ drawn independently from the same model.

## Setup and graph models

In [1]:
import itertools
import numpy as np
from collections import Counter
from math import sqrt


# ----------------------------------------------------------------------
# graph utilities
# ----------------------------------------------------------------------
def all_perms(n):
    """All n! permutations as an (n!, n) index array."""
    return np.array(list(itertools.permutations(range(n))), dtype=np.int16)


def er_graph(n, p, rng):
    """Undirected Erdos-Renyi G(n, p), no self-loops, as a 0/1 adjacency matrix."""
    M = (rng.random((n, n)) < p).astype(np.int8)
    M = np.triu(M, 1)
    return M + M.T

# ----------------------------------------------------------------------
# random graph models  (all return an n x n 0/1 int8 adjacency matrix)
# ----------------------------------------------------------------------
def _nx_to_adj(G, n, rng):
    import networkx as nx
    A = nx.to_numpy_array(G, nodelist=range(n), dtype=np.int8)
    perm = rng.permutation(n)             # random labelling (generators order nodes by arrival/block)
    return A[np.ix_(perm, perm)]


def ba_graph(n, m, rng):
    """Barabasi-Albert: each new node attaches to m existing nodes, prob. proportional to degree."""
    import networkx as nx
    return _nx_to_adj(nx.barabasi_albert_graph(n, m, seed=int(rng.integers(2**31))), n, rng)


def sbm_graph(n, n_blocks, p_in, p_out, rng):
    """Stochastic block model with n_blocks (near-)equal blocks."""
    blocks = np.arange(n) % n_blocks
    same = blocks[:, None] == blocks[None, :]
    M = (rng.random((n, n)) < np.where(same, p_in, p_out)).astype(np.int8)
    M = np.triu(M, 1)
    M = M + M.T
    perm = rng.permutation(n)
    return M[np.ix_(perm, perm)]


# name -> (description, sampler(n, rng))
MODELS = {
    "ER":  ("Erdos-Renyi p=0.5",                  lambda n, rng: er_graph(n, 0.5, rng)),
    "BA":  ("Barabasi-Albert m=2",                lambda n, rng: ba_graph(n, 2, rng)),
    "SBM": ("SBM 2 blocks, p_in=0.7, p_out=0.2",  lambda n, rng: sbm_graph(n, 2, 0.7, 0.2, rng)),
}


def flip_k_edges(A, k, rng):
    """Flip k distinct off-diagonal entries (add or delete k edges)."""
    n = A.shape[0]
    iu = np.triu_indices(n, 1)
    idx = rng.choice(len(iu[0]), size=k, replace=False)
    B = A.copy()
    for t in idx:
        i, j = iu[0][t], iu[1][t]
        B[i, j] ^= 1
        B[j, i] = B[i, j]
    return B

## Exact criterion check (brute force over all $n!$ relabellings)

In [2]:
def orbit_scan(A, AH, P, chunk=20000):
    """
    Enumerate all relabellings S of H and return
        (|O|, GED)   where  O = argmin_S ||A - S A_H S^T||_1.
    Chunked to bound peak memory.
    """
    best, count = None, 0
    for s in range(0, len(P), chunk):
        Q = P[s:s + chunk]
        M = AH[Q[:, :, None], Q[:, None, :]]           # (chunk, n, n) relabelled copies
        d = np.abs(M - A[None]).sum(axis=(1, 2))       # entrywise L1 = 2 * edge edit distance
        m = int(d.min())
        if best is None or m < best:
            best, count = m, int((d == m).sum())
        elif m == best:
            count += int((d == m).sum())
    return count, best


def aut_size(A, P, chunk=20000):
    """|Aut(G)| by brute force."""
    tot = 0
    for s in range(0, len(P), chunk):
        Q = P[s:s + chunk]
        M = A[Q[:, :, None], Q[:, None, :]]
        tot += int((np.abs(M - A[None]).sum(axis=(1, 2)) == 0).sum())
    return tot


def satisfiable(A, AH, P):
    """
    Check both versions of the criterion for the pair (A, A'=AH).

    labelled   : A' in N(A)  as given (no relabelling of A'):
                   ||A - A'||_1 < min_{P not in Aut(G)} ||P A P^T - A'||_1.
                 Since ||P A P^T - A'|| = ||A - S A' S^T|| with S = P^T, this holds
                 iff d0 := ||A - A'|| equals the optimum (GED) AND the optimum is
                 attained only by S in Aut(G)  (i.e. |O| = |Aut(G)|).
    unlabelled : G_{A'} in N(G_A), i.e. SOME relabelling P A' P^T lies in N(A)
                 <=> |O| = |Aut(G)|.
    """
    O, ged = orbit_scan(A, AH, P)
    aG = aut_size(A, P)
    aH = aut_size(AH, P)
    d0 = int(np.abs(A.astype(np.int16) - AH).sum())
    unl = (O == aG)
    return dict(
        ok=unl,                       # unlabelled criterion (kept for backward compat.)
        ok_unlabelled=unl,
        ok_labelled=bool(unl and d0 == ged),
        d0=d0,                        # ||A - A'||_1 in the given labelling
        n_copies=O // aG,             # number of distinct copies of G at optimal distance
        ged=ged,
        autG=aG,
        autH=aH,
        forced=(aH > aG),             # symmetry-forced tie: excluded a priori
    )

## Experiment

In [3]:
def wilson(k, n, z=1.96):
    """Wilson score interval for a binomial proportion."""
    if n == 0:
        return (float("nan"), float("nan"))
    ph = k / n
    d = 1 + z * z / n
    c = (ph + z * z / (2 * n)) / d
    h = z * sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))


def run_pairs(n, sampler, trials, seed, k=None):
    """
    Draw `trials` pairs (A, A') and count how many satisfy each criterion.
      k is None -> independent: A, A' drawn independently from the same model.
      k = int   -> k-flip:      A' = A with k random edge flips.
    """
    rng = np.random.default_rng(seed)
    P = all_perms(n)
    lab = unl = 0
    dens = []
    for _ in range(trials):
        A = sampler(n, rng)
        AH = sampler(n, rng) if k is None else flip_k_edges(A, k, rng)
        r = satisfiable(A, AH, P)
        lab += r["ok_labelled"]
        unl += r["ok_unlabelled"]
        dens.append(A.sum() / (n * (n - 1)))
    return dict(lab=lab, unl=unl, trials=trials, density=float(np.mean(dens)))


def fmt_row(model, group, n, res):
    t = res["trials"]
    l, u = res["lab"], res["unl"]
    llo, lhi = wilson(l, t)
    ulo, uhi = wilson(u, t)
    return (f"{model:>5} {group:>12} {n:>3} {res['density']:>7.2f} {t:>7}   "
            f"{100*l/t:>6.1f}% [{100*llo:5.1f},{100*lhi:5.1f}]   "
            f"{100*u/t:>6.1f}% [{100*ulo:5.1f},{100*uhi:5.1f}]")

## Run the simulation

In [4]:
N = 8              # brute force over n! = 40320 permutations
KS = [1, 2, 3]     # number of edge flips
TRIALS = 500       # pairs per (model, group)
SEED = 0

for name, (desc, _) in MODELS.items():
    print(f"{name}: {desc}")

results = []       # (model, group, res)
for mi, (name, (_, sampler)) in enumerate(MODELS.items()):
    groups = [(f"{k}-flip", k) for k in KS] + [("independent", None)]
    for gi, (g, k) in enumerate(groups):
        res = run_pairs(N, sampler, TRIALS, seed=SEED + 10000 * N + 100 * mi + gi, k=k)
        results.append((name, g, res))
        print(f"  {name:>4} {g:>12}: labelled {100*res['lab']/TRIALS:5.1f}%   "
              f"unlabelled {100*res['unl']/TRIALS:5.1f}%", flush=True)

ER: Erdos-Renyi p=0.5
BA: Barabasi-Albert m=2
SBM: SBM 2 blocks, p_in=0.7, p_out=0.2


    ER       1-flip: labelled  71.0%   unlabelled  71.0%


    ER       2-flip: labelled  37.8%   unlabelled  39.6%


    ER       3-flip: labelled  13.8%   unlabelled  18.8%


    ER  independent: labelled   0.0%   unlabelled   9.0%


    BA       1-flip: labelled  71.4%   unlabelled  71.4%


... (3 outputs left out)

   SBM       1-flip: labelled  69.2%   unlabelled  69.2%


   SBM       2-flip: labelled  32.4%   unlabelled  34.6%


   SBM       3-flip: labelled  14.0%   unlabelled  20.4%


   SBM  independent: labelled   0.0%   unlabelled   9.8%


## Final table
Each cell: **labelled % / unlabelled %** of pairs satisfying the criterion (95% Wilson CI in the detailed table below).

In [ ]:
# from IPython.display import Markdown, display

# groups = [f"{k}-flip" for k in KS] + ["independent"]
# lookup = {(m, g): r for m, g, r in results}
# pct = lambda x, t: f"{100*x/t:.1f}"

# lines = ["| model | density | " + " | ".join(groups) + " |",
#          "|---|---|" + "---|" * len(groups)]
# for m in MODELS:
#     dens = lookup[(m, groups[0])]["density"]
#     cells = [f"{pct(lookup[(m,g)]['lab'], TRIALS)} / {pct(lookup[(m,g)]['unl'], TRIALS)}" for g in groups]
#     lines.append(f"| {m} | {dens:.2f} | " + " | ".join(cells) + " |")
# display(Markdown(f"**n = {N}, {TRIALS} pairs per cell — labelled % / unlabelled %**\n\n" + "\n".join(lines)))

# detail = ["| model | group | labelled % [95% CI] | unlabelled % [95% CI] |", "|---|---|---|---|"]
# for m, g, r in results:
#     (ll, lh), (ul, uh) = wilson(r["lab"], TRIALS), wilson(r["unl"], TRIALS)
#     detail.append(f"| {m} | {g} | {pct(r['lab'],TRIALS)} [{100*ll:.1f}, {100*lh:.1f}] "
#                   f"| {pct(r['unl'],TRIALS)} [{100*ul:.1f}, {100*uh:.1f}] |")
# display(Markdown("\n".join(detail)))

**n = 8, 500 pairs per cell — labelled % / unlabelled %**

| model | density | 1-flip | 2-flip | 3-flip | independent |
|---|---|---|---|---|---|
| ER | 0.50 | 71.0 / 71.0 | 37.8 / 39.6 | 13.8 / 18.8 | 0.0 / 9.0 |
| BA | 0.43 | 71.4 / 71.4 | 40.0 / 41.4 | 16.4 / 20.6 | 0.0 / 12.2 |
| SBM | 0.42 | 69.2 / 69.2 | 32.4 / 34.6 | 14.0 / 20.4 | 0.0 / 9.8 |

| model | group | labelled % [95% CI] | unlabelled % [95% CI] |
|---|---|---|---|
| ER | 1-flip | 71.0 [66.9, 74.8] | 71.0 [66.9, 74.8] |
| ER | 2-flip | 37.8 [33.7, 42.1] | 39.6 [35.4, 44.0] |
| ER | 3-flip | 13.8 [11.1, 17.1] | 18.8 [15.6, 22.5] |
| ER | independent | 0.0 [0.0, 0.8] | 9.0 [6.8, 11.8] |
| BA | 1-flip | 71.4 [67.3, 75.2] | 71.4 [67.3, 75.2] |
| BA | 2-flip | 40.0 [35.8, 44.4] | 41.4 [37.2, 45.8] |
| BA | 3-flip | 16.4 [13.4, 19.9] | 20.6 [17.3, 24.4] |
| BA | independent | 0.0 [0.0, 0.8] | 12.2 [9.6, 15.4] |
| SBM | 1-flip | 69.2 [65.0, 73.1] | 69.2 [65.0, 73.1] |
| SBM | 2-flip | 32.4 [28.4, 36.6] | 34.6 [30.6, 38.9] |
| SBM | 3-flip | 14.0 [11.2, 17.3] | 20.4 [17.1, 24.2] |
| SBM | independent | 0.0 [0.0, 0.8] | 9.8 [7.5, 12.7] |